In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="qWD6Bt11JBvzDZdxZUQb")
project = rf.workspace("roboflow-universe-projects").project("fall-detection-ca3o8")
dataset = project.version(4).download("yolov5")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 114.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Fall-Detection-4 in yolov5pytorch:: 100%|██████████| 21586/21586 [00:02<00:00, 8144.34it/s] 


In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jinjinjaraannechoi","key":"bf2248b223c3e36b131a608f13338dc9"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d uttejkumarkandagatla/fall-detection-dataset
!unzip -q fall-detection-dataset.zip -d fall_raw

Dataset URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
License(s): ODbL-1.0
  0% 0.00/49.7M [00:00<?, ?B/s]
100% 49.7M/49.7M [00:00<00:00, 1.30GB/s]


In [5]:
!ls fall_raw/

fall_dataset


In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="qWD6Bt11JBvzDZdxZUQb")
project = rf.workspace("new-workspace-qfcus").project("le2i-with-upright-and-fall")
version = project.version(2)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Le2i-with-upright-and-fall-2 in yolov5pytorch:: 100%|██████████| 16260/16260 [00:01<00:00, 10764.57it/s]


In [7]:
!kaggle datasets download -d uttejkumarkandagatla/fall-detection-dataset
!unzip fall-detection-dataset.zip -d /content/fall_raw2

Dataset URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
License(s): ODbL-1.0
fall-detection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  fall-detection-dataset.zip
  inflating: /content/fall_raw2/fall_dataset/images/train/fall001.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall002.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall003.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall004.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall005.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall006.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall007.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall008.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall009.jpg  
  inflating: /content/fall_raw2/fall_dataset/images/train/fall010.jpg  
  inflating: /content/f

In [8]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17564, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 17564 (delta 35), reused 7 (delta 7), pack-reused 17507 (from 3)
Receiving objects: 100% (17564/17564), 16.64 MiB | 31.90 MiB/s, done.
Resolving deltas: 100% (12029/12029), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.5 MB/s eta 0:00:00


In [13]:
import os, glob, shutil, hashlib, random
from pathlib import Path

random.seed(42)

# ==== 사용자 경로/설정 ====
SRC_ROOT = Path("/content/datasets")  # 각 데이터셋 폴더들이 이 아래에 있다고 가정
OUT = Path("/content/unified_dataset")
SPLIT_RATIO = 0.8  # train 80% / val 20%

# 데이터셋별 fall 클래스 id 설정 (모르면 기본 0)
FALL_IDS = {
    "le2i": {0},
    "uttej": {0},
    "robofall": {0},
    "falldetection": {0},
}

# ==== 전역 카운터 ====
total_pos = {"train":0, "val":0}
total_neg = {"train":0, "val":0}

# ==== 함수 ====
def list_images(dir_: Path):
    exts = (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
    return [Path(p) for p in glob.glob(str(dir_ / "**/*"), recursive=True) if Path(p).suffix.lower() in exts]

def load_yolo_labels(lbl_path: Path):
    if not lbl_path.exists(): return []
    with open(lbl_path, "r") as f:
        return [ln.strip() for ln in f if ln.strip()]

def keep_fall_only(lines, fall_ids_set):
    kept = []
    for ln in lines:
        ps = ln.split()
        if len(ps) < 5:
            continue
        try:
            cid = int(ps[0])
        except:
            continue
        if cid in fall_ids_set:
            ps[0] = "0"  # 통일: fall → 0
            kept.append(" ".join(ps))
    return kept

def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)

def make_new_stem(ds_name, img_path: Path):
    h = hashlib.md5(str(img_path).encode()).hexdigest()[:8]
    return f"{ds_name}_{h}_{img_path.stem}"

def pair_and_copy(split, img_list, lbl_dirs, ds_name, fall_ids):
    global total_pos, total_neg  # 수정 포인트!

    pos_cnt = neg_cnt = 0
    for img in img_list:
        lbl_path = None
        for ld in lbl_dirs:
            cand = ld / (img.stem + ".txt")
            if cand.exists():
                lbl_path = cand
                break

        if lbl_path:
            lines = load_yolo_labels(lbl_path)
            fall_only = keep_fall_only(lines, fall_ids)
        else:
            fall_only = []

        # 새 이름/경로
        new_stem = make_new_stem(ds_name, img)
        out_img = OUT/f"images/{split}/{new_stem}{img.suffix.lower()}"
        out_lbl = OUT/f"labels/{split}/{new_stem}.txt"

        safe_copy(img, out_img)
        if fall_only:
            with open(out_lbl, "w") as fw:
                fw.write("\n".join(fall_only)+"\n")
            pos_cnt += 1
        else:
            neg_cnt += 1

    total_pos[split] += pos_cnt
    total_neg[split] += neg_cnt
    print(f"[{ds_name}] {split}: fall {pos_cnt}장, negatives {neg_cnt}장")

# ==== 실행 ====
for p in [OUT/"images/train", OUT/"images/val", OUT/"labels/train", OUT/"labels/val"]:
    p.mkdir(parents=True, exist_ok=True)

datasets = [d for d in SRC_ROOT.iterdir() if d.is_dir()]
print("found datasets:", [d.name for d in datasets])

for ds in datasets:
    ds_name = ds.name
    fall_ids = set(FALL_IDS.get(ds_name, {0}))

    img_dirs = [ds/"images"]
    lbl_dirs = [ds/"labels"]

    if not img_dirs[0].exists():
        print(f"[{ds_name}] skip: no images/")
        continue

    imgs = list_images(img_dirs[0])
    random.shuffle(imgs)
    cut = int(len(imgs)*SPLIT_RATIO)
    train_imgs = imgs[:cut]
    val_imgs   = imgs[cut:]

    pair_and_copy("train", train_imgs, lbl_dirs, ds_name, fall_ids)
    pair_and_copy("val",   val_imgs,   lbl_dirs, ds_name, fall_ids)

print("\n==== SUMMARY ====")
for sp in ["train","val"]:
    n_img = len(list((OUT/f"images/{sp}").glob("*.*")))
    n_lbl = len(list((OUT/f"labels/{sp}").glob("*.txt")))
    print(f"{sp}: images={n_img}, labels={n_lbl}, fall_imgs={total_pos[sp]}, negatives={total_neg[sp]}")

# data.yaml 작성
yaml_text = f"""
train: {OUT}/images/train
val:   {OUT}/images/val

nc: 1
names: ['fall']
"""
(OUT/"fall_1cls.yaml").write_text(yaml_text)
print("\nWrote:", OUT/"fall_1cls.yaml")


found datasets: ['uttej', 'falldetection', 'le2i', 'robofall']
[uttej] train: fall 0장, negatives 388장
[uttej] val: fall 0장, negatives 97장
[falldetection] skip: no images/
[le2i] skip: no images/
[robofall] train: fall 0장, negatives 388장
[robofall] val: fall 0장, negatives 97장

==== SUMMARY ====
train: images=776, labels=0, fall_imgs=0, negatives=776
val: images=194, labels=0, fall_imgs=0, negatives=194

Wrote: /content/unified_dataset/fall_1cls.yaml


In [16]:
# 각 데이터셋 폴더 안 구조 확인
!ls -R /content/datasets/uttej | head -n 30
!ls -R /content/datasets/robofall | head -n 30
!ls -R /content/datasets/le2i | head -n 30
!ls -R /content/datasets/falldetection | head -n 30


/content/datasets/uttej:
images
labels

/content/datasets/uttej/images:
train
val

/content/datasets/uttej/images/train:
fall001.jpg
fall002.jpg
fall003.jpg
fall004.jpg
fall005.jpg
fall006.jpg
fall007.jpg
fall008.jpg
fall009.jpg
fall010.jpg
fall011.jpg
fall012.jpg
fall013.jpg
fall014.jpg
fall015.jpg
fall016.jpg
fall017.jpg
fall018.jpg
fall019.jpg
fall020.jpg
fall021.jpg
/content/datasets/robofall:
images
labels

/content/datasets/robofall/images:
train
val

/content/datasets/robofall/images/train:
fall001.jpg
fall002.jpg
fall003.jpg
fall004.jpg
fall005.jpg
fall006.jpg
fall007.jpg
fall008.jpg
fall009.jpg
fall010.jpg
fall011.jpg
fall012.jpg
fall013.jpg
fall014.jpg
fall015.jpg
fall016.jpg
fall017.jpg
fall018.jpg
fall019.jpg
fall020.jpg
fall021.jpg
/content/datasets/le2i:
data.yaml
README.dataset.txt
README.roboflow.txt
test
train
valid

/content/datasets/le2i/test:
images
labels

/content/datasets/le2i/test/images:
000021_jpg.rf.017c725b026af648118f399da6b2be7b.jpg
000028_jpg.rf.d63e56c55

In [17]:
# uttej/robofall 라벨이 split별 폴더에 있는지
!find /content/datasets/uttej/labels -type f -name "*.txt" | head -n 5
!find /content/datasets/robofall/labels -type f -name "*.txt" | head -n 5

# le2i/falldetection는 train/valid/test 안에 labels가 있어야 함
!find /content/datasets/le2i -type d -name "labels" | head -n 5
!find /content/datasets/falldetection -type d -name "labels" | head -n 5


/content/datasets/uttej/labels/val/fall050.txt
/content/datasets/uttej/labels/val/not fallen033.txt
/content/datasets/uttej/labels/val/fall024.txt
/content/datasets/uttej/labels/val/fall065.txt
/content/datasets/uttej/labels/val/not fallen015.txt
/content/datasets/robofall/labels/val/fall050.txt
/content/datasets/robofall/labels/val/not fallen033.txt
/content/datasets/robofall/labels/val/fall024.txt
/content/datasets/robofall/labels/val/fall065.txt
/content/datasets/robofall/labels/val/not fallen015.txt
/content/datasets/le2i/train/labels
/content/datasets/le2i/test/labels
/content/datasets/le2i/valid/labels
/content/datasets/falldetection/train/labels
/content/datasets/falldetection/test/labels
/content/datasets/falldetection/valid/labels


In [18]:
import glob
from collections import Counter

def class_hist(root):
    cnt=Counter()
    for p in glob.glob(root+"/**/*.txt", recursive=True):
        for ln in open(p):
            ln=ln.strip()
            if not ln: continue
            a=ln.split()[0]
            if a.isdigit(): cnt[int(a)] += 1
    return cnt

for name in ["uttej","robofall","le2i","falldetection"]:
    print(name, class_hist(f"/content/datasets/{name}"))


uttej Counter({0: 285, 1: 143, 2: 139})
robofall Counter({0: 285, 1: 143, 2: 139})
le2i Counter({0: 4064, 1: 4060})
falldetection Counter({0: 10790})


In [19]:
import os, glob, shutil, hashlib, random
from pathlib import Path

random.seed(42)

SRC_ROOT = Path("/content/datasets")
OUT = Path("/content/unified_dataset")
for p in [OUT/"images/train", OUT/"images/val", OUT/"labels/train", OUT/"labels/val"]:
    p.mkdir(parents=True, exist_ok=True)

# 필요시 여기서 fall id 수정 (기본 0)
FALL_IDS = {
    "uttej": {0},
    "robofall": {0},
    "le2i": {0},
    "falldetection": {0},
}

def list_images(dir_):
    exts = (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
    return [Path(p) for p in glob.glob(str(dir_/"**/*"), recursive=True) if Path(p).suffix.lower() in exts]

def load_labels(p):
    if not p.exists(): return []
    return [ln.strip() for ln in open(p) if ln.strip()]

def keep_fall_only(lines, fall_ids):
    kept=[]
    for ln in lines:
        ps=ln.split()
        if len(ps)<5 or not ps[0].isdigit(): continue
        if int(ps[0]) in fall_ids:
            ps[0]="0"
            kept.append(" ".join(ps))
    return kept

def safe_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)

def new_stem(ds_name, img):
    h = hashlib.md5(str(img).encode()).hexdigest()[:8]
    return f"{ds_name}_{h}_{img.stem}"

def merge_type_classic(ds):  # uttej, robofall
    ds_name = ds.name
    fall_ids = FALL_IDS.get(ds_name, {0})
    for split in ["train","val"]:
        img_dir = ds/"images"/split
        lbl_dir = ds/"labels"/split
        if not img_dir.exists() or not lbl_dir.exists():
            print(f"[{ds_name}] skip {split}: missing {img_dir} or {lbl_dir}")
            continue
        imgs = list_images(img_dir)
        pos=neg=0
        for img in imgs:
            lbl = lbl_dir/(img.stem+".txt")
            lines = load_labels(lbl)
            fall_only = keep_fall_only(lines, fall_ids) if lines else []
            stem = new_stem(ds_name, img)
            out_img = OUT/f"images/{split}/{stem}{img.suffix.lower()}"
            out_lbl = OUT/f"labels/{split}/{stem}.txt"
            safe_copy(img, out_img)
            if fall_only:
                with open(out_lbl,"w") as f: f.write("\n".join(fall_only)+"\n")
                pos+=1
            else:
                neg+=1
        print(f"[{ds_name}] {split}: fall {pos}장, negatives {neg}장")

def merge_type_roboflow(ds):  # le2i, falldetection
    ds_name = ds.name
    fall_ids = FALL_IDS.get(ds_name, {0})
    # train → train, valid → val (test는 val에 합쳐도 됨)
    mapping = [("train","train"), ("valid","val")]  # 필요시 ("test","val") 추가
    for src_split, dst_split in mapping:
        img_dir = ds/src_split/"images"
        lbl_dir = ds/src_split/"labels"
        if not img_dir.exists() or not lbl_dir.exists():
            print(f"[{ds_name}] skip {src_split}: missing {img_dir} or {lbl_dir}")
            continue
        imgs = list_images(img_dir)
        pos=neg=0
        for img in imgs:
            lbl = lbl_dir/(img.stem+".txt")
            lines = load_labels(lbl)
            fall_only = keep_fall_only(lines, fall_ids) if lines else []
            stem = new_stem(ds_name, img)
            out_img = OUT/f"images/{dst_split}/{stem}{img.suffix.lower()}"
            out_lbl = OUT/f"labels/{dst_split}/{stem}.txt"
            safe_copy(img, out_img)
            if fall_only:
                with open(out_lbl,"w") as f: f.write("\n".join(fall_only)+"\n")
                pos+=1
            else:
                neg+=1
        print(f"[{ds_name}] {src_split}->{dst_split}: fall {pos}장, negatives {neg}장")

# 실행
for ds in [d for d in SRC_ROOT.iterdir() if d.is_dir()]:
    if (ds/"images"/"train").exists() and (ds/"labels"/"train").exists():
        merge_type_classic(ds)
    elif (ds/"train"/"images").exists() and (ds/"train"/"labels").exists():
        merge_type_roboflow(ds)
    else:
        print(f"[{ds.name}] 알 수 없는 구조. 건너뜀.")

print("\n==== SUMMARY ====")
for sp in ["train","val"]:
    n_img = len(list((OUT/f"images/{sp}").glob("*.*")))
    n_lbl = len(list((OUT/f"labels/{sp}").glob("*.txt")))
    print(f"{sp}: images={n_img}, labels={n_lbl}")

# data.yaml (1클래스)
yaml_text = f"""
train: {OUT}/images/train
val:   {OUT}/images/val

nc: 1
names: ['fall']
"""
(OUT/"fall_1cls.yaml").write_text(yaml_text)
print("\nWrote:", OUT/"fall_1cls.yaml")


[uttej] train: fall 213장, negatives 161장
[uttej] val: fall 72장, negatives 39장
[falldetection] train->train: fall 9438장, negatives 0장
[falldetection] valid->val: fall 899장, negatives 0장
[le2i] train->train: fall 3843장, negatives 3828장
[le2i] valid->val: fall 151장, negatives 152장
[robofall] train: fall 213장, negatives 161장
[robofall] val: fall 72장, negatives 39장

==== SUMMARY ====
train: images=18038, labels=13707
val: images=1577, labels=1194

Wrote: /content/unified_dataset/fall_1cls.yaml


In [20]:
# 이미지/라벨 개수 확인
!ls /content/unified_dataset/images/train | wc -l
!ls /content/unified_dataset/images/val   | wc -l
!ls /content/unified_dataset/labels/train | wc -l
!ls /content/unified_dataset/labels/val   | wc -l


18038
1577
13707
1194


In [24]:
%cd /content/yolov5
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 80 \
  --data /content/unified_dataset/fall_1cls.yaml \
  --weights yolov5s.pt \
  --project fall_project \
  --hyp hyp.scratch-low.yaml \
  --cos-lr \
  --patience 20


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
      77/79      4.61G    0.01836   0.008678          0         27        640:  79% 896/1128 [01:11<00:19, 11.98it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      77/79      4.61G    0.01836   0.008678          0         26        640:  79% 896/1128 [01:11<00:19, 11.98it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      77/79      4.61G    0.01836    0.00868          0         31        640:  80% 898/1128 [01:12<00:18, 12.11it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      77/79      4.61G    0.01836   0.00868

In [29]:
!python /content/yolov5/detect.py \
  --weights /content/yolov5/fall_project/exp/weights/best.pt \
  --source /content/test_images \
  --img 960 \
  --conf 0.15 \
  --save-txt --save-conf \
  --name quick_batch


detect: weights=['/content/yolov5/fall_project/exp/weights/best.pt'], source=/content/test_images, data=data/coco128.yaml, imgsz=[960, 960], conf_thres=0.15, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=quick_batch, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-430-g459d8bf0 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
image 1/1 /content/test_images/79578_22932_5047.jpg: 640x960 1 fall, 50.3ms
Speed: 0.7ms pre-process, 50.3ms inference, 130.8ms NMS per image at shape (1, 3, 960, 960)
Results saved to runs/detect/quick_batch5
1 labels saved to runs/detect/quick_batch5/labels
